# kNN implementation


---


### 01. Library Installation


In [ ]:
%pip install -qq imbalanced-learn matplotlib numpy pandas scikit-learn seaborn


### 02. Library Imports


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from imblearn.over_sampling import RandomOverSampler

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import learning_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from math import sqrt

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)


### 03. Data Loading and Preprocessing


Looking at the dataset that will be processed, this is the **MAGIC Gamma Telescope Dataset** which contains data from a ground-based atmospheric Cherenkov gamma telescope. The dataset is used for classification of high energy gamma particles from hadrons (background noise).

**Dataset characteristics:**
- **Total samples**: 19,020 observations
- **Features**: 10 continuous variables measuring telescope imaging parameters
- **Target**: Binary classification (Gamma particles vs Hadrons)
    - Class 1: Gamma particles (signal)
    - Class 0: Hadrons (background)

**Feature descriptions:**
- `fLength`: Major axis of ellipse [mm]
- `fWidth`: Minor axis of ellipse [mm]  
- `fSize`: 10-log of sum of content of all pixels [in #phot]
- `fConc`: Ratio of sum of two highest pixels over fSize [ratio]
- `fConc1`: Ratio of highest pixel over fSize [ratio]
- `fAsym`: Distance from highest pixel to center, projected onto major axis [mm]
- `fM3Long`: 3rd root of third moment along major axis [mm]
- `fM3Trans`: 3rd root of third moment along minor axis [mm]
- `fAlpha`: Angle of major axis with vector to origin [deg]
- `fDist`: Distance from origin to center of ellipse [mm]

The goal is to distinguish between gamma-ray showers (which are of astrophysical interest) and hadronic showers initiated by cosmic rays in the upper atmosphere (which represent background noise).


In [ ]:
# Importing and checking the dataframe

dataframe = pd.read_csv('../../datasets/magic_gamma_telescope/magic04.data', header = None)
dataframe.head()


In [ ]:
# Renaming columns accordingly to the dataset documentation

columns_name = ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1', 'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class']
dataframe.columns = columns_name
dataframe.head()


In [ ]:
# Mapping the classes to numerical values

dataframe['class'] = dataframe['class'].map({'g': 1, 'h': 0})
dataframe.head()


### 04. Data Visualization


In [ ]:
for label in columns_name[:-1]:
    plt.figure(figsize = (10, 6))

    plt.hist(dataframe[dataframe['class'] == 0][label], label = 'Hadron (h)', density = True, alpha = 0.5, color = 'blue')
    plt.hist(dataframe[dataframe['class'] == 1][label], label = 'Gamma (g)', density = True, alpha = 0.5, color = 'orange')

    plt.title(f'Distribution of {label} by Class')
    plt.xlabel(label)
    plt.ylabel('Probability')
    plt.legend(title = 'Class')
    plt.grid()

    plt.show()


In [ ]:
# Checking the correlation matrix

plt.figure(figsize = (10, 8))
sns.heatmap(dataframe.corr(), annot = True, cmap = 'coolwarm', vmin = -1, vmax = 1)
plt.title('Correlation Matrix')
plt.show()


### 05. Dataset Splitting and Scaling


In [ ]:
# Shuffling the dataframe

dataframe = dataframe.sample(frac = 1, random_state = 42).reset_index(drop = True)


In [ ]:
dataframe.info()


In [ ]:
dataframe.head()


In [ ]:
# Defining a function to scale the data and oversample if necessary

def scale_data(dataframe, oversample = False):
    features = dataframe.columns[:-1]
    target = dataframe.columns[-1]

    X = dataframe[features].values
    y = dataframe[target].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    if oversample:
        ros = RandomOverSampler(random_state = 42)
        X_resampled, y_resampled = ros.fit_resample(X_scaled, y)
        dataframe_scaled = np.hstack((X_resampled, np.reshape(y_resampled, (-1, 1))))
        return dataframe_scaled, X_resampled, y_resampled
    else:
        dataframe_scaled = np.hstack((X_scaled, np.reshape(y, (-1, 1))))
        return dataframe_scaled, X_scaled, y


In [ ]:
# Defining the train, validation and test datasets sizes

train_size = 0.7
validation_size = 0.15
test_size = 0.15


In [ ]:
# Defining the train, validation and test datasets

train_dataset = dataframe[:int(train_size * len(dataframe))]
validation_dataset = dataframe[int(train_size * len(dataframe)):int((train_size + validation_size) * len(dataframe))]
test_dataset = dataframe[int((train_size + validation_size) * len(dataframe)):]

print('\nDatasets sizes before scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


In [ ]:
# Scaling and oversampling the datasets

train_dataset, X_train, y_train = scale_data(train_dataset, oversample = True)
validation_dataset, X_validation, y_validation = scale_data(validation_dataset, oversample = False)
test_dataset, X_test, y_test = scale_data(test_dataset, oversample = False)

print('\nDatasets sizes after scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


### 06. kNN Implementation and Evaluation


In [ ]:
# kNN implementation

for number_neighbors in range(1, 5, 2):
    knn_model = KNeighborsClassifier(n_neighbors = number_neighbors)
    knn_model.fit(X_train, y_train)

    y_predictions_validation = knn_model.predict(X_validation)
    y_predictions_test = knn_model.predict(X_test)

    print(f'\nNumber of Neighbors: {number_neighbors}')
    print('Validation Set Classification Report:')
    print(classification_report(y_validation, y_predictions_validation))

    print('Test Set Classification Report:')
    print(classification_report(y_test, y_predictions_test))


In [ ]:
# Using the rule of thumb to define the number of neighbors

def optimal_k(data):
    return int(sqrt(len(data)))

optimal_neighbors = optimal_k(X_train)
print(f'\nOptimal number of neighbors (k) using the rule of thumb: {optimal_neighbors}')

knn_model = KNeighborsClassifier(n_neighbors = optimal_neighbors)
knn_model.fit(X_train, y_train)

y_predictions_validation = knn_model.predict(X_validation)
y_predictions_test = knn_model.predict(X_test)


In [ ]:
# General classification reports

print(f'\nNumber of Neighbors: {optimal_neighbors}')

print('Validation Set Classification Report:')
print(classification_report(y_validation, y_predictions_validation))

print('Test Set Classification Report:')
print(classification_report(y_test, y_predictions_test))


In [ ]:
# Confusion matrix

confusion_matrix_validation = confusion_matrix(y_validation, y_predictions_validation)
confusion_matrix_test = confusion_matrix(y_test, y_predictions_test)

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_validation, annot = True, fmt = 'd', cmap = 'Blues', cbar = False)
plt.title('Confusion Matrix - Validation Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_test, annot = True, fmt = 'd', cmap = 'Greens', cbar = False)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
# Getting probability predictions for additional metrics

y_pred_proba_validation = knn_model.predict_proba(X_validation)
y_pred_proba_test = knn_model.predict_proba(X_test)


In [ ]:
# Basic accuracy metrics

accuracy_validation = accuracy_score(y_validation, y_predictions_validation)
accuracy_test = accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Accuracy: {accuracy_validation:.4f}')
print(f'Test Set Accuracy: {accuracy_test:.4f}')


In [ ]:
# Precision metrics

precision_validation = precision_score(y_validation, y_predictions_validation, average = 'weighted')
precision_test = precision_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Precision (weighted): {precision_validation:.4f}')
print(f'Test Set Precision (weighted): {precision_test:.4f}')

# Class-specific precision
precision_per_class_validation = precision_score(y_validation, y_predictions_validation, average = None)
precision_per_class_test = precision_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Precision:')
print(f'\tHadron (0): {precision_per_class_validation[0]:.4f}')
print(f'\tGamma (1): {precision_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific Precision:')
print(f'\tHadron (0): {precision_per_class_test[0]:.4f}')
print(f'\tGamma (1): {precision_per_class_test[1]:.4f}')


In [ ]:
# Recall metrics

recall_validation = recall_score(y_validation, y_predictions_validation, average = 'weighted')
recall_test = recall_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Recall (weighted): {recall_validation:.4f}')
print(f'Test Set Recall (weighted): {recall_test:.4f}')

# Class-specific recall
recall_per_class_validation = recall_score(y_validation, y_predictions_validation, average = None)
recall_per_class_test = recall_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Recall:')
print(f'\tHadron (0): {recall_per_class_validation[0]:.4f}')
print(f'\tGamma (1): {recall_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific Recall:')
print(f'\tHadron (0): {recall_per_class_test[0]:.4f}')
print(f'\tGamma (1): {recall_per_class_test[1]:.4f}')


In [ ]:
# F1-Score metrics

f1_validation = f1_score(y_validation, y_predictions_validation, average = 'weighted')
f1_test = f1_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set F1-Score (weighted): {f1_validation:.4f}')
print(f'Test Set F1-Score (weighted): {f1_test:.4f}')

# Class-specific F1-Score
f1_per_class_validation = f1_score(y_validation, y_predictions_validation, average = None)
f1_per_class_test = f1_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific F1-Score:')
print(f'\tHadron (0): {f1_per_class_validation[0]:.4f}')
print(f'\tGamma (1): {f1_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific F1-Score:')
print(f'\tHadron (0): {f1_per_class_test[0]:.4f}')
print(f'\tGamma (1): {f1_per_class_test[1]:.4f}')


In [ ]:
# Balanced accuracy

balanced_acc_validation = balanced_accuracy_score(y_validation, y_predictions_validation)
balanced_acc_test = balanced_accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Balanced Accuracy: {balanced_acc_validation:.4f}')
print(f'Test Set Balanced Accuracy: {balanced_acc_test:.4f}')


In [ ]:
# Matthews Correlation Coefficient (MCC)

mcc_validation = matthews_corrcoef(y_validation, y_predictions_validation)
mcc_test = matthews_corrcoef(y_test, y_predictions_test)

print(f'Validation Set Matthews Correlation Coefficient: {mcc_validation:.4f}')
print(f'Test Set Matthews Correlation Coefficient: {mcc_test:.4f}')


In [ ]:
# Cohen's Kappa

kappa_validation = cohen_kappa_score(y_validation, y_predictions_validation)
kappa_test = cohen_kappa_score(y_test, y_predictions_test)

print(f'Validation Set Cohen\'s Kappa: {kappa_validation:.4f}')
print(f'Test Set Cohen\'s Kappa: {kappa_test:.4f}')


In [ ]:
# ROC AUC

roc_auc_validation = roc_auc_score(y_validation, y_pred_proba_validation[:, 1])
roc_auc_test = roc_auc_score(y_test, y_pred_proba_test[:, 1])

print(f'Validation Set ROC AUC: {roc_auc_validation:.4f}')
print(f'Test Set ROC AUC: {roc_auc_test:.4f}')


In [ ]:
# Average Precision (PR AUC)

avg_precision_validation = average_precision_score(y_validation, y_pred_proba_validation[:, 1])
avg_precision_test = average_precision_score(y_test, y_pred_proba_test[:, 1])

print(f'Validation Set Average Precision (PR AUC): {avg_precision_validation:.4f}')
print(f'Test Set Average Precision (PR AUC): {avg_precision_test:.4f}')


In [ ]:
# Log Loss

logloss_validation = log_loss(y_validation, y_pred_proba_validation)
logloss_test = log_loss(y_test, y_pred_proba_test)

print(f'Validation Set Log Loss: {logloss_validation:.4f}')
print(f'Test Set Log Loss: {logloss_test:.4f}')


In [ ]:
# ROC Curve - Validation Set

fpr_val, tpr_val, _ = roc_curve(y_validation, y_pred_proba_validation[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(fpr_val, tpr_val, color = 'darkorange', lw = 2, label = f'ROC curve (AUC = {roc_auc_validation:.4f})')
plt.plot([0, 1], [0, 1], color = 'navy', lw = 2, linestyle = '--', label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Validation Set')
plt.legend(loc = "lower right")
plt.grid(True)
plt.show()


In [ ]:
# ROC Curve - Test Set

fpr_test, tpr_test, _ = roc_curve(y_test, y_pred_proba_test[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(fpr_test, tpr_test, color = 'darkorange', lw = 2, label = f'ROC curve (AUC = {roc_auc_test:.4f})')
plt.plot([0, 1], [0, 1], color = 'navy', lw = 2, linestyle = '--', label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set')
plt.legend(loc = "lower right")
plt.grid(True)
plt.show()


In [ ]:
# Precision-Recall Curve - Validation Set

precision_val, recall_val, _ = precision_recall_curve(y_validation, y_pred_proba_validation[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(recall_val, precision_val, color = 'blue', lw = 2, label = f'PR curve (AP = {avg_precision_validation:.4f})')
plt.axhline(y = np.mean(y_validation), color = 'red', linestyle = '--', label = f'Random Classifier (AP = {np.mean(y_validation):.4f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Validation Set')
plt.legend(loc = "lower left")
plt.grid(True)
plt.show()


In [ ]:
# Precision-Recall Curve - Test Set

precision_test, recall_test, _ = precision_recall_curve(y_test, y_pred_proba_test[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(recall_test, precision_test, color = 'blue', lw = 2, label = f'PR curve (AP = {avg_precision_test:.4f})')
plt.axhline(y = np.mean(y_test), color = 'red', linestyle = '--', label = f'Random Classifier (AP = {np.mean(y_test):.4f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Test Set')
plt.legend(loc = "lower left")
plt.grid(True)
plt.show()


In [ ]:
# Cross-validation with accuracy

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
cv_accuracy_scores = cross_val_score(knn_model, X_train, y_train, cv = cv, scoring = 'accuracy')

print(f'Cross-validation Accuracy:')
print(f'\tMean: {cv_accuracy_scores.mean():.4f} (+/- {cv_accuracy_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_accuracy_scores]}')


In [ ]:
# Cross-validation with precision

cv_precision_scores = cross_val_score(knn_model, X_train, y_train, cv = cv, scoring = 'precision_weighted')

print(f'Cross-validation Precision (weighted):')
print(f'\tMean: {cv_precision_scores.mean():.4f} (+/- {cv_precision_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_precision_scores]}')


In [ ]:
# Cross-validation with recall

cv_recall_scores = cross_val_score(knn_model, X_train, y_train, cv = cv, scoring = 'recall_weighted')

print(f'Cross-validation Recall (weighted):')
print(f'\tMean: {cv_recall_scores.mean():.4f} (+/- {cv_recall_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_recall_scores]}')


In [ ]:
# Cross-validation with F1-Score

cv_f1_scores = cross_val_score(knn_model, X_train, y_train, cv = cv, scoring = 'f1_weighted')

print(f'Cross-validation F1-Score (weighted):')
print(f'\tMean: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_f1_scores]}')


In [ ]:
# Cross-validation with ROC AUC

cv_roc_auc_scores = cross_val_score(knn_model, X_train, y_train, cv = cv, scoring = 'roc_auc')

print(f'Cross-validation ROC AUC:')
print(f'\tMean: {cv_roc_auc_scores.mean():.4f} (+/- {cv_roc_auc_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_roc_auc_scores]}')


In [ ]:
# Error analysis - Validation Set

misclassified_mask_validation = y_validation != y_predictions_validation
misclassified_indices_validation = np.where(misclassified_mask_validation)[0]

print(f'Validation Set Error Analysis:')
print(f'Total misclassified samples: {len(misclassified_indices_validation)} out of {len(y_validation)}')
print(f'Error rate: {len(misclassified_indices_validation)/len(y_validation)*100:.2f}%')

# Analyze misclassification by true class
for true_class in [0, 1]:
    class_name = 'Hadron' if true_class == 0 else 'Gamma'
    true_class_mask = y_validation == true_class
    misclass_in_class = np.sum(misclassified_mask_validation & true_class_mask)
    total_in_class = np.sum(true_class_mask)

    print(f'{class_name} particles (Class {true_class}):')
    print(f'\tMisclassified: {misclass_in_class}/{total_in_class} ({misclass_in_class/total_in_class*100:.2f}%)')


In [ ]:
# Error analysis - Test Set

misclassified_mask_test = y_test != y_predictions_test
misclassified_indices_test = np.where(misclassified_mask_test)[0]

print(f'Test Set Error Analysis:')
print(f'Total misclassified samples: {len(misclassified_indices_test)} out of {len(y_test)}')
print(f'Error rate: {len(misclassified_indices_test)/len(y_test)*100:.2f}%')

# Analyze misclassification by true class
for true_class in [0, 1]:
    class_name = 'Hadron' if true_class == 0 else 'Gamma'
    true_class_mask = y_test == true_class
    misclass_in_class = np.sum(misclassified_mask_test & true_class_mask)
    total_in_class = np.sum(true_class_mask)

    print(f'{class_name} particles (Class {true_class}):')
    print(f'\tMisclassified: {misclass_in_class}/{total_in_class} ({misclass_in_class/total_in_class*100:.2f}%)')


In [ ]:
# Confidence analysis

# Misclassified samples confidence
misclass_confidences_validation = np.max(y_pred_proba_validation[misclassified_mask_validation], axis = 1)
correct_confidences_validation = np.max(y_pred_proba_validation[misclassified_mask_validation], axis = 1)

misclass_confidences_test = np.max(y_pred_proba_test[misclassified_mask_test], axis = 1)
correct_confidences_test = np.max(y_pred_proba_test[misclassified_mask_test], axis = 1)

print('Confidence Analysis:')
print(f'Validation Set:')
print(f'\tAverage confidence of misclassified samples: {misclass_confidences_validation.mean():.4f}')
print(f'\tAverage confidence of correct predictions: {correct_confidences_validation.mean():.4f}')

print(f'Test Set:')
print(f'\tAverage confidence of misclassified samples: {misclass_confidences_test.mean():.4f}')
print(f'\tAverage confidence of correct predictions: {correct_confidences_test.mean():.4f}')


In [ ]:
# Learning curve analysis

train_sizes = np.linspace(0.1, 1.0, 10)

train_sizes_abs, train_scores, val_scores = learning_curve(
    knn_model, X_train, y_train, train_sizes = train_sizes, cv = 5, 
    scoring = 'accuracy', random_state = 42, n_jobs = -1
)

train_mean = np.mean(train_scores, axis = 1)
train_std = np.std(train_scores, axis = 1)
val_mean = np.mean(val_scores, axis = 1)
val_std = np.std(val_scores, axis = 1)

plt.figure(figsize = (10, 6))
plt.plot(train_sizes_abs, train_mean, 'o-', color = 'blue', label = 'Training Accuracy')
plt.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha = 0.1, color = 'blue')

plt.plot(train_sizes_abs, val_mean, 'o-', color = 'red', label = 'Validation Accuracy')
plt.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha = 0.1, color = 'red')

plt.xlabel('Training Set Size')
plt.ylabel('Accuracy Score')
plt.title(f'Learning Curve - kNN (k = {optimal_neighbors})')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# K-value optimization analysis

k_range = range(1, 21)
k_values = []
train_accuracy = []
val_accuracy = []
train_f1 = []
val_f1 = []
val_roc_auc = []

for k in k_range:
    # Train model with current k
    knn_temp = KNeighborsClassifier(n_neighbors = k)
    knn_temp.fit(X_train, y_train)

    # Predictions
    train_pred = knn_temp.predict(X_train)
    val_pred = knn_temp.predict(X_validation)
    val_pred_proba = knn_temp.predict_proba(X_validation)

    # Calculate metrics
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_validation, val_pred)
    train_f1_score = f1_score(y_train, train_pred, average = 'weighted')
    val_f1_score = f1_score(y_validation, val_pred, average = 'weighted')
    val_auc = roc_auc_score(y_validation, val_pred_proba[:, 1])

    # Store results
    k_values.append(k)
    train_accuracy.append(train_acc)
    val_accuracy.append(val_acc)
    train_f1.append(train_f1_score)
    val_f1.append(val_f1_score)
    val_roc_auc.append(val_auc)

    # Print details for some k values
    if k <= 10 or k % 5 == 0:
        print(f'k={k:2d}: Val Acc={val_acc:.4f}, Val F1={val_f1_score:.4f}, Val AUC={val_auc:.4f}')

# Find optimal k for different metrics
best_k_acc = k_values[np.argmax(val_accuracy)]
best_k_f1 = k_values[np.argmax(val_f1)]
best_k_auc = k_values[np.argmax(val_roc_auc)]

print(f'\nOptimal k values:')
print(f'\tBest k for Accuracy: {best_k_acc} (score: {max(val_accuracy):.4f})')
print(f'\tBest k for F1-Score: {best_k_f1} (score: {max(val_f1):.4f})')
print(f'\tBest k for ROC-AUC: {best_k_auc} (score: {max(val_roc_auc):.4f})')

# Using the rule of thumb k for final evaluation
print(f'\nRule of thumb k: {optimal_neighbors}')
knn_temp = KNeighborsClassifier(n_neighbors = optimal_neighbors)
knn_temp.fit(X_train, y_train)

# Predictions
train_pred = knn_temp.predict(X_train)
val_pred = knn_temp.predict(X_validation)
val_pred_proba = knn_temp.predict_proba(X_validation)

# Calculate metrics
train_acc = accuracy_score(y_train, train_pred)
val_acc = accuracy_score(y_validation, val_pred)
train_f1_score = f1_score(y_train, train_pred, average = 'weighted')
val_f1_score = f1_score(y_validation, val_pred, average = 'weighted')
val_auc = roc_auc_score(y_validation, val_pred_proba[:, 1])

# Store results
k_values.append(optimal_neighbors)
train_accuracy.append(train_acc)
val_accuracy.append(val_acc)
train_f1.append(train_f1_score)
val_f1.append(val_f1_score)
val_roc_auc.append(val_auc)

print(f'k={optimal_neighbors:2d}: Val Acc={val_acc:.4f}, Val F1={val_f1_score:.4f}, Val AUC={val_auc:.4f}')



In [ ]:
# K-value analysis visualization

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize = (15, 12))

# Accuracy plot
ax1.plot(k_values, train_accuracy, 'o-', label = 'Training Accuracy', color = 'blue')
ax1.plot(k_values, val_accuracy, 'o-', label = 'Validation Accuracy', color = 'red')
ax1.set_xlabel('k Value')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs k Value')
ax1.legend()
ax1.grid(True)

# F1-Score plot
ax2.plot(k_values, train_f1, 'o-', label = 'Training F1', color = 'blue')
ax2.plot(k_values, val_f1, 'o-', label = 'Validation F1', color = 'red')
ax2.set_xlabel('k Value')
ax2.set_ylabel('F1-Score')
ax2.set_title('F1-Score vs k Value')
ax2.legend()
ax2.grid(True)

# ROC-AUC plot
ax3.plot(k_values, val_roc_auc, 'o-', label = 'Validation ROC-AUC', color = 'green')
ax3.set_xlabel('k Value')
ax3.set_ylabel('ROC-AUC')
ax3.set_title('ROC-AUC vs k Value')
ax3.legend()
ax3.grid(True)

# Bias-Variance tradeoff visualization
train_val_diff = np.array(train_accuracy) - np.array(val_accuracy)
ax4.plot(k_values, train_val_diff, 'o-', label = 'Training - Validation Gap', color = 'purple')
ax4.axhline(y = 0, color = 'black', linestyle = '--', alpha = 0.5)
ax4.set_xlabel('k Value')
ax4.set_ylabel('Accuracy Difference')
ax4.set_title('Bias-Variance Tradeoff (Training - Validation Gap)')
ax4.legend()
ax4.grid(True)

plt.tight_layout()
plt.show()


### 07. Summary and Interpretation of Additional Metrics

The additional metrics provide deeper insights into your kNN model performance:

**Basic Classification Metrics:**
- **Balanced Accuracy**: Accounts for class imbalance better than regular accuracy
- **Matthews Correlation Coefficient (MCC)**: Measures correlation between observed and predicted classifications (-1 to +1, where +1 is perfect)
- **Cohen's Kappa**: Inter-rater reliability statistic that accounts for agreement by chance
- **Class-specific metrics**: Individual precision, recall, and F1 for each class

**Probability-based Metrics:**
- **ROC-AUC**: Area under the ROC curve, measures discriminative ability
- **Average Precision (PR-AUC)**: Area under Precision-Recall curve, better for imbalanced datasets
- **Log Loss**: Quantifies the uncertainty of predictions based on probabilities

**Analysis Techniques:**
- **Cross-validation**: Provides robust estimates with confidence intervals
- **Error Analysis**: Identifies patterns in misclassified samples and confidence levels
- **Learning Curve**: Shows if more data would improve performance
- **K-value Optimization**: Systematic approach to find optimal k value

**ROC and PR Curves:**
- ROC curves show trade-off between sensitivity and specificity
- PR curves are more informative for imbalanced datasets
- Higher AUC values indicate better performance

These metrics help you:
1. **Assess model reliability** through cross-validation
2. **Understand failure cases** through error analysis
3. **Optimize hyperparameters** systematically
4. **Choose appropriate metrics** for your specific problem
5. **Compare models** objectively across multiple dimensions
